In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_raw = pd.read_csv("/Users/sahra/Documents/Sideprosjekt/cycle-tracker/cycle-tracker/data/raw/cycle_data.csv")
df = df_raw.copy()
df['Date'] = pd.to_datetime(df['Date'])
print(df.shape)
print(df.head())

(1731, 16)
        Date  Temperature Menstruation  LH test  Pregnancy test Had sex  \
0 2021-04-01          NaN          NaN      NaN             NaN     YES   
1 2021-04-02          NaN          NaN      NaN             NaN      NO   
2 2021-04-15        36.83          NaN      NaN             NaN      NO   
3 2021-04-16        36.65          NaN      NaN             NaN      NO   
4 2021-04-17        36.62          NaN      NaN             NaN     NaN   

                Notes Skipped  Source  \
0                 NaN     NaN     NaN   
1  Scared im pregnant     NaN     NaN   
2                 NaN     NaN     NaN   
3                 NaN     NaN     NaN   
4                 NaN     NaN     NaN   

                                           Data Flag Menstruation Quantity  \
0                                                NaN                   NaN   
1  PAIN_OVULATION,PAIN_BACKACHE,MOOD_SENSITIVE,MO...                   NaN   
2     PAIN_SORE_BREASTS,MOOD_ENERGETIC,MOOD_STRESSED     

In [5]:
df['Temperature'] = df['Temperature'].fillna(method='bfill').fillna(method='ffill')
print(df['Temperature'].isna().sum(), "manglende temperaturer igjen")

0 manglende temperaturer igjen


/var/folders/wg/p2zzj8s50v31rz_ysr4446x80000gn/T/ipykernel_27159/1228073310.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Temperature'] = df['Temperature'].fillna(method='bfill').fillna(method='ffill')


In [6]:
df['is_menstruation'] = df['Menstruation'] == 'MENSTRUATION'
df['cycle_number'] = (df['is_menstruation'] & ~df['is_menstruation'].shift(1, fill_value=False)).cumsum()
df['temp_rolling_mean'] = df['Temperature'].rolling(window=5, min_periods=1).mean()
df['temp_shift'] = df['Temperature'] - df['temp_rolling_mean'].shift(6)
df['possible_ovulation'] = df['temp_shift'] > 0.2
df['phase'] = 'follicular'
df.loc[df['possible_ovulation'], 'phase'] = 'ovulation'
df.loc[df['possible_ovulation'].shift(1, fill_value=False), 'phase'] = 'luteal'

In [7]:
train = df[df['Date'] < '2025-01-01'].copy()
test = df[df['Date'] >= '2025-01-01'].copy()
print(f"Train: {len(train)} rader")
print(f"Test: {len(test)} rader")

Train: 1320 rader
Test: 411 rader


In [16]:
cycle_day = []
current_cycle_day = 0
for i, row in df.iterrows():
    if row['is_menstruation'] and (i == 0 or not df.loc[i-1, 'is_menstruation']):
        current_cycle_day = 1
    else:
        current_cycle_day += 1
    cycle_day.append(current_cycle_day)
df['cycle_day'] = cycle_day
train = df[df['Date'] < '2025-01-01'].copy()
test = df[df['Date'] >= '2025-01-01'].copy()

def calculate_coverline(cycle_df):
    follicular = cycle_df[cycle_df['cycle_day'] <= 10]['Temperature']
    if len(follicular) == 0:
        return None
    return follicular.median() + 0.1

def detect_ovulation_confirmed(cycle_df, coverline):
    temps = cycle_df['Temperature'].values
    confirmed_day = None
    for i in range(2, len(temps)):
        if (temps[i] > coverline and 
            temps[i-1] > coverline and 
            temps[i-2] > coverline):
            confirmed_day = i
            break
    return confirmed_day

def predict_fertile(cycle_df):
    coverline = calculate_coverline(cycle_df)
    if coverline is None:
        return cycle_df.assign(fertile=True)
    confirmed_day = detect_ovulation_confirmed(cycle_df, coverline)
    cycle_df = cycle_df.copy()
    if confirmed_day is None:
        cycle_df['fertile'] = True
    else:
        cycle_df['fertile'] = True
        cycle_df.iloc[confirmed_day + 1:, cycle_df.columns.get_loc('fertile')] = False
    cycle_df['coverline'] = coverline
    cycle_df['ovulation_confirmed_day'] = confirmed_day
    return cycle_df

In [17]:
# Kjør predict_fertile på treningsdata
results = []
for cycle_num in train['cycle_number'].unique():
    cycle = train[train['cycle_number'] == cycle_num].copy().reset_index(drop=True)
    if len(cycle) > 10:
        result = predict_fertile(cycle)
        results.append(result)

all_cycles = pd.concat(results).reset_index(drop=True)

print(f"Totalt: {len(all_cycles)} dager")
print(f"Fertile dager: {all_cycles['fertile'].sum()}")
print(f"Ikke-fertile dager: {(~all_cycles['fertile']).sum()}")
print(f"Andel fertile: {all_cycles['fertile'].mean():.1%}")

Totalt: 1296 dager
Fertile dager: 839
Ikke-fertile dager: 457
Andel fertile: 64.7%


In [ ]:
first_cycle = train[train['cycle_number'] == 2].copy().reset_index(drop=True)
result = predict_fertile(first_cycle)
print(result[['D'
'ate', 'Temperature', 'cycle_day', 'fertile']].to_string())

         Date  Temperature  cycle_day  fertile
0  2021-04-24        36.35          1     True
1  2021-04-25        36.38          2     True
2  2021-04-26        36.12          3     True
3  2021-04-27        36.21          4     True
4  2021-04-28        36.16          5     True
5  2021-04-29        36.36          6     True
6  2021-04-30        36.39          7     True
7  2021-05-01        36.19          8     True
8  2021-05-02        36.62          9     True
9  2021-05-03        36.50         10     True
10 2021-05-04        36.33         11     True
11 2021-05-05        36.15         12     True
12 2021-05-06        36.38         13     True
13 2021-05-07        36.52         14     True
14 2021-05-08        36.68         15     True
15 2021-05-09        36.55         16     True
16 2021-05-10        36.23         17    False
17 2021-05-11        36.45         18    False
18 2021-05-12        36.52         19    False
19 2021-05-13        36.91         20    False
20 2021-05-14

In [23]:
results = []
for cycle_num in train['cycle_number'].unique():
    cycle = train[train['cycle_number'] == cycle_num].copy().reset_index(drop=True)
    if len(cycle) > 10:
        result = predict_fertile(cycle)
        results.append(result)

all_cycles = pd.concat(results).reset_index(drop=True)

print(f"Totalt: {len(all_cycles)} dager")
print(f"Fertile dager: {all_cycles['fertile'].sum()}")
print(f"Ikke-fertile dager: {(~all_cycles['fertile']).sum()}")
print(f"Andel fertile: {all_cycles['fertile'].mean():.1%}")

Totalt: 1296 dager
Fertile dager: 839
Ikke-fertile dager: 457
Andel fertile: 64.7%
